> **Research record.** Cleaned copy of `800-research-users-locations/2000-google-pois-flags.ipynb` (data checks, dead ends and unlabelled plots removed; outputs cleared).
> Builds the Google-POI flags (category, location, reviews, visitor behaviour, hourly signal stability, place2vec similarity) that feed the PU model. The same pipeline as one SQL script is `02-google-pois-flags-pipeline.sql`.
>
> Requires the authors' BigQuery project and the full GPS dataset, which are not public; the notebook is kept as a record of the analysis behind the paper. Run it with its own folder as the working directory and `settings.ini` at the repository root. For a runnable walk-through on synthetic data see `demo/`.

# Description
* -- Google POIs remote-work flags pipeline
* -- Output: phd-habidatum.phd_pois.google_pois_flags
* -- Period for behavior flag: 2021-01-01 to 2023-12-31
* -- Each flag is updated independently with its own query.


* Review/atmosphere flags from phd_pois.google_3rd_places_rw_review    

* Groundtruth flag from phd_pois.third_rw_places_groundtruth via ST_DWITHIN(..., 40) + best EDIT_DISTANCE
* Location flag from phd_datasets.geohash_features exclusions (universities/colleges/schools/medical/transport == 0 and flag_work_zone = 0)
* Behavior flag from phd_datasets.pois_visits_monthly_19022026, worker_type='remote_workers', period 2021-01-01 to 2023-12-31, with rw_share >= 0.20

In [ ]:
import os
import sys
from pyproj import datadir

# Ensure PROJ uses the database from this conda env
env_proj = os.path.join(sys.prefix, "share", "proj")
if not os.path.isdir(env_proj):
    raise RuntimeError(f"PROJ data dir not found at {env_proj}")

# Set both env vars and pyproj's internal data dir
os.environ["PROJ_DATA"] = env_proj
os.environ["PROJ_LIB"] = env_proj
datadir.set_data_dir(env_proj)

print("Using PROJ data from:", env_proj)
print("pyproj data dir:", datadir.get_data_dir())

In [ ]:
import sys
from pathlib import Path
prepare_data_path = Path().absolute().parent
if str(prepare_data_path) not in sys.path:
    sys.path.insert(0, str(prepare_data_path))

In [ ]:
from geofunctions import utils

In [ ]:

# -- ============================================================================
# -- 0) Base table (Google only)
# -- ============================================================================

utils.read_query_to_dataframe("""CREATE OR REPLACE TABLE `phd-habidatum.phd_pois.google_pois_features`
CLUSTER BY geometry AS

  SELECT
    CAST(g.id AS STRING) AS id,
    g.displayname.text AS name,
    LOWER(g.primarytype) AS primarytype,
    g.geometry,
    gs.geohash,
    gsa.yishuv_stat_2022,
    sa.new_neighborhood_type AS census_new_neighborhood_type
  FROM `phd-habidatum.phd_pois.google_3rdplaces_cities` g
  JOIN `phd-habidatum.phd_adm.geohashes_with_sa_gushdan` gs
    ON ST_WITHIN(g.geometry, gs.geometry)
  LEFT JOIN `phd-habidatum.phd_adm.geohashes_with_sa_gushdan_context` gsa
    USING (geohash)
  LEFT JOIN `phd-habidatum.phd_datasets.sa_neigborhoods_types_new` sa
    ON CAST(sa.yishuv_stat_2022 AS STRING) = CAST(gsa.yishuv_stat_2022 AS STRING)

""")

In [ ]:
# -- ============================================================================
# -- 1) Flag: POI category suits remote work
# --    (SELECT * + new feature
# -- ============================================================================
utils.read_query_to_dataframe("""CREATE OR REPLACE TABLE `phd-habidatum.phd_pois.google_pois_features`
CLUSTER BY geometry AS
SELECT
  * ,
  CASE
    WHEN primarytype IS NULL THEN NULL
    WHEN primarytype IN (
      'cafe', 'coffee_shop', 'tea_house', 'bakery', 'book_store', 'library',
      'community_center', 'cultural_center', 'museum', 'dessert_restaurant',
      'breakfast_restaurant', 'brunch_restaurant', 'cat_cafe', 'dog_cafe'
    ) THEN 1
    ELSE 0
  END AS flag_category_suits_rw,
FROM `phd-habidatum.phd_pois.google_pois_features`;
"""
)

In [ ]:
# -- ============================================================================
# -- 2) Flag: Location suits remote work (not office/university)
# --    (SELECT * + new feature, no UPDATE)
# -- ============================================================================
utils.read_query_to_dataframe("""
CREATE OR REPLACE TABLE `phd-habidatum.phd_pois.google_pois_features`
CLUSTER BY geometry AS
WITH transport_hubs_threshold AS (
  select APPROX_QUANTILES(cat_transport_count, 100)[OFFSET(99)] AS transport_hubs
  FROM `phd-habidatum.phd_datasets.geohash_features`
  where cat_transport_count>0
),
location_calc AS (
  SELECT
    p.id,
    CASE
      WHEN (COALESCE(gf.flag_work_zone, 0) = 1 AND COALESCE(gf.bld_residential_count, 0) = 0)
        OR COALESCE(sa.main_function_txt, '') = 'תעשיה' THEN 1
      ELSE 0
    END AS flag_work_zone,
    CASE
      WHEN COALESCE(gf.count_universities, 0)
         + COALESCE(gf.count_colleges, 0)
         + COALESCE(gf.count_medical_institutions, 0) = 0 THEN 0
      ELSE 1
    END AS flag_work_institute,
    PERCENT_RANK() OVER (
    ORDER BY COALESCE(count_bc_corp_offices_250m_google, 0)
  ) AS office_probability_risk,
    CASE
      WHEN cat_transport_count >= th.transport_hubs THEN 1
      ELSE 0
    END AS flag_transport_hub
  FROM `phd-habidatum.phd_pois.google_pois_features` p
  JOIN phd-habidatum.phd_adm.geohashes_with_sa_gushdan sa
    using(geohash)
  LEFT JOIN `phd-habidatum.phd_datasets.geohash_features` gf
    USING (geohash)
  CROSS JOIN transport_hubs_threshold th
)
SELECT
  p.* except(flag_location_suits_rw,office_probability_risk),
  CASE
    WHEN l.flag_work_zone + l.flag_work_institute = 0 THEN 1
     ELSE 0
  END AS flag_location_suits_rw,
  flag_transport_hub flag_transport_hub_risk,
  l.office_probability_risk -- 0.9 seems to be a good threshold
FROM `phd-habidatum.phd_pois.google_pois_features` p
LEFT JOIN location_calc l
  USING (id);
""")

In [ ]:
# -- ============================================================================
# -- 3) Flags: Review evidence (direct RW keyword evidence),  Atmosphere suits remote work

# -- ============================================================================
utils.read_query_to_dataframe("""CREATE OR REPLACE TABLE `phd-habidatum.phd_pois.google_pois_features`
CLUSTER BY geometry AS
WITH review_calc AS (
  SELECT
    p.id,
    cast(flag_direct_keyword as int) flag_keyword_review_rw,
    cast(flag_atmosphere as int) flag_atmosphere_suits_rw,
    c.rating,
    c.userratingcount,
    c.livemusic,
    c.servesdessert,
    c.servescoffee,
    c.restroom,
    c.pricerange
  FROM `phd-habidatum.phd_pois.google_pois_features` p
  LEFT JOIN `phd-habidatum.phd_pois.google_3rd_places_rw_review` r
    USING (id)
  JOIN `phd-habidatum.phd_pois.google_3rdplaces_cities` c
    USING (id)
)
select g.*except(flag_keyword_review_rw,flag_atmosphere_suits_rw,rating,
    userratingcount,
    livemusic,
    servesdessert,
    servescoffee,
    restroom,
    pricerange), 
coalesce(r.flag_keyword_review_rw,0) flag_keyword_review_rw,
coalesce(r.flag_atmosphere_suits_rw,0) flag_atmosphere_suits_rw, 
 r.rating,
    r.userratingcount,
    r.livemusic,
    r.servesdessert,
    r.servescoffee,
    r.restroom,
    r.pricerange
from `phd-habidatum.phd_pois.google_pois_features` g
left join review_calc r
using(id)
""")

In [ ]:
# -- ============================================================================
# -- 4) USERS BEHAVIOR: Visits share of remote workers
# -- ============================================================================

utils.read_query_to_dataframe("""CREATE OR REPLACE TABLE `phd-habidatum.phd_pois.google_pois_features`
CLUSTER BY geometry AS
WITH poi_avg AS (
  SELECT
    poi_id,
    SAFE_DIVIDE(n_remote_work_visits, n_visits) AS avg_rw_visits_share_after2022,
    SAFE_DIVIDE(n_visits_after_home, n_visits) AS share_visits_after_home,
    SAFE_DIVIDE(n_visits_after_work, n_visits) AS share_visits_after_work,
    visit_frequency,
    avg_dist_home,
    avg_dist_home/avg_dist_work as avg_dist_home_work_ratio,
    share_dwells_home_lt_1km,
    share_weekday,
    share_friday,
    avg_duration_hours_weekday,
    n_visits_weekday
  FROM `phd-habidatum.phd_datasets.pois_visits_metrics_2022_2023`
  WHERE n_visits_weekday > 0
    AND worker_type = 'all_workers'
),
poi_rw AS (
  SELECT
    poi_id,
    n_visits_weekday,
    avg_duration_hours_weekday,
    visit_frequency,
    avg_dist_home,
    share_dwells_home_lt_1km
  FROM `phd-habidatum.phd_datasets.pois_visits_metrics_2022_2023`
  WHERE n_visits_weekday > 0
    AND worker_type = 'remote_workers'
),
m as (
SELECT
  poi_id as id,
  avg_rw_visits_share_after2022,
  p.share_visits_after_home,
  p.share_visits_after_work,
  p.avg_dist_home,
  p.avg_dist_home_work_ratio,
  p.visit_frequency,
  p.share_dwells_home_lt_1km,
  p.share_weekday,
  p.share_friday,
  p.avg_duration_hours_weekday,
  p.n_visits_weekday,
  r.n_visits_weekday as n_visits_weekday_rw,
  r.avg_duration_hours_weekday as avg_duration_hours_weekday_rw,
  r.visit_frequency as visit_frequency_rw,
  r.avg_dist_home as avg_dist_home_rw,
  r.share_dwells_home_lt_1km as share_dwells_home_lt_1km_rw,
  r.n_visits_weekday/p.n_visits_weekday as remote_workers_visits_share_weekday,
  PERCENT_RANK() OVER (ORDER BY avg_rw_visits_share_after2022) AS avg_rw_visits_share_percentile_0_1,
  
FROM poi_avg p
join poi_rw r
using(poi_id)
)
select p.*except(avg_rw_visits_share_after2022,
  avg_dist_home,
  avg_dist_home_work_ratio,
  visit_frequency,
  share_dwells_home_lt_1km,
  share_weekday,
  share_friday,
  avg_duration_hours_weekday,
  n_visits_weekday,
  n_visits_weekday_rw,
  avg_duration_hours_weekday_rw,
  visit_frequency_rw,
  avg_dist_home_rw,
  share_dwells_home_lt_1km_rw,
  remote_workers_visits_share_weekday,
  avg_rw_visits_share_percentile_0_1,
  flag_rw_visits_share_after_2022),
m.*except(id),
case when m.avg_rw_visits_share_after2022>=0.2 then 1 else 0 end as flag_rw_visits_share_after_2022
from `phd-habidatum.phd_pois.google_pois_features` p
left join m
using(id)
""")

In [ ]:
#--========================================================
# -- 5) Flag: Signals dont drop too much during the day
#--========================================================

utils.read_query_to_dataframe("""CREATE OR REPLACE TABLE `phd-habidatum.phd_pois.google_pois_features`
CLUSTER BY geometry AS
WITH poi_avg AS (
  SELECT
    poi_id,
    weekday_type,
    period,
    hour,
    day_normalized_hour_share,
    n_signals
  FROM `phd-habidatum.phd_users_locations.pois_hourly_counts_post_covid`
),
m as (
SELECT
  poi_id as id,
  max(n_signals) as max_n_signals,
  min(day_normalized_hour_share) AS min_norm_share_10_15
FROM poi_avg
where weekday_type='weekday' and period='post_2022' and hour between 10 and 15
group by poi_id
)
select p.* except(min_norm_share_10_15,max_n_signals, flag_office_substitute_daily_dynamic), 
m.min_norm_share_10_15,
m.max_n_signals,
case when m.min_norm_share_10_15>=0.2 and m.max_n_signals>=10 then 1 else 0 end as flag_office_substitute_daily_dynamic, 
from `phd-habidatum.phd_pois.google_pois_features` p
left join m
using(id)
""")

### 2nd version: std , recommended by Daniel

In [ ]:
MIN_HOURS_COUNT=5

In [ ]:
#--========================================================
# -- 5b) Features: hourly_signal_std (post_2022, weekday)
#--========================================================

utils.read_query_to_dataframe(f"""
CREATE OR REPLACE TABLE `phd-habidatum.phd_pois.google_pois_features`
CLUSTER BY geometry AS
WITH period_totals AS (
  SELECT
    period,
    SUM(n_signals) AS total_signals_in_dataset_all_hours
  FROM `phd-habidatum.phd_users_locations.pois_hourly_counts_post_covid`
  WHERE period = 'post_2022'
  GROUP BY period
),

hourly AS (
  SELECT
    h.poi_id,
    h.period,
    h.hour,
    SAFE_DIVIDE(h.n_signals, pt.total_signals_in_dataset_all_hours) AS hour_share
  FROM `phd-habidatum.phd_users_locations.pois_hourly_counts_post_covid` h
  INNER JOIN period_totals pt
    USING (period)
  WHERE h.period = 'post_2022'
    AND h.n_signals IS NOT NULL
    AND pt.total_signals_in_dataset_all_hours > 0
    and h.weekday_type = 'weekday' 
),

with_keyword AS (
  SELECT
    h.poi_id,
    h.hour,
    h.hour_share,
    LOWER(p.keyword) AS keyword
  FROM hourly h
  INNER JOIN `phd-habidatum.phd_pois.google_osm_frsq_pois_wth_duplicates` p
    ON p.id = h.poi_id
  WHERE p.keyword IS NOT NULL
),

dispersion AS (
  SELECT
    poi_id,
    keyword,
    STDDEV_SAMP(hour_share) AS hourly_signal_std
  FROM with_keyword
  GROUP BY poi_id, keyword
  HAVING COUNT(*) >= {MIN_HOURS_COUNT}
),

dispersion_weekday AS (
  SELECT
    poi_id,
    hourly_signal_std,
    PERCENT_RANK() OVER (
      PARTITION BY keyword
      ORDER BY hourly_signal_std
    ) AS hourly_signal_std_percentile_by_keyword
  FROM dispersion
)

SELECT
  p.*except(hourly_signal_std, hourly_signal_std_percentile_by_keyword),
  d.hourly_signal_std,
  d.hourly_signal_std_percentile_by_keyword
FROM `phd-habidatum.phd_pois.google_pois_features` p
LEFT JOIN dispersion_weekday d
  ON p.id = d.poi_id
""")

## Cosine similarity to work from place2vec

In [ ]:
# by neighborhood type is commented because of too many nulls
#avg_cosine_similarity_work_morning_residential_remote, avg_cosine_similarity_work_late_morning_residential_remote, avg_cosine_similarity_work_afternoon_residential_remote, avg_cosine_similarity_work_morning_residential_central, avg_cosine_similarity_work_late_morning_residential_central, avg_cosine_similarity_work_afternoon_residential_central, avg_cosine_similarity_work_morning_cbd, avg_cosine_similarity_work_late_morning_cbd, avg_cosine_similarity_work_afternoon_cbd, avg_cosine_similarity_work_morning_satellite_city, avg_cosine_similarity_work_late_morning_satellite_city, avg_cosine_similarity_work_afternoon_satellite_city, avg_cosine_similarity_work_morning_same_neighborhood_type, avg_cosine_similarity_work_late_morning_same_neighborhood_type, avg_cosine_similarity_work_afternoon_same_neighborhood_type, 
utils.read_query_to_dataframe(f"""
CREATE OR REPLACE TABLE `phd-habidatum.phd_pois.google_pois_features`
CLUSTER BY geometry AS
with m as (
select 
poi_id,
avg(p.cosine_similarity_morning) as avg_cosine_similarity_work_morning,
avg(p.cosine_similarity_late_morning) as avg_cosine_similarity_work_late_morning,
avg(p.cosine_similarity_afternoon) as avg_cosine_similarity_work_afternoon,

# avg(case when work_neighborhood_type = 'residential_remote' or work_neighborhood_type is null then p.cosine_similarity_morning else null end) as avg_cosine_similarity_work_morning_residential_remote,
# avg(case when work_neighborhood_type = 'residential_remote' or work_neighborhood_type is null then p.cosine_similarity_late_morning else null end) as avg_cosine_similarity_work_late_morning_residential_remote,
# avg(case when work_neighborhood_type = 'residential_remote' or work_neighborhood_type is null then p.cosine_similarity_afternoon else null end) as avg_cosine_similarity_work_afternoon_residential_remote,

# avg(case when work_neighborhood_type = 'residential_central' then p.cosine_similarity_morning else null end) as avg_cosine_similarity_work_morning_residential_central,
# avg(case when work_neighborhood_type = 'residential_central' then p.cosine_similarity_late_morning else null end) as avg_cosine_similarity_work_late_morning_residential_central,
# avg(case when work_neighborhood_type = 'residential_central' then p.cosine_similarity_afternoon else null end) as avg_cosine_similarity_work_afternoon_residential_central,

# avg(case when work_neighborhood_type = 'CBD' then p.cosine_similarity_morning else null end) as avg_cosine_similarity_work_morning_cbd,
# avg(case when work_neighborhood_type = 'CBD' then p.cosine_similarity_late_morning else null end) as avg_cosine_similarity_work_late_morning_cbd,
# avg(case when work_neighborhood_type = 'CBD' then p.cosine_similarity_afternoon else null end) as avg_cosine_similarity_work_afternoon_cbd,

# avg(case when work_neighborhood_type = 'satellite_city' then p.cosine_similarity_morning else null end) as avg_cosine_similarity_work_morning_satellite_city,
# avg(case when work_neighborhood_type = 'satellite_city' then p.cosine_similarity_late_morning else null end) as avg_cosine_similarity_work_late_morning_satellite_city,
# avg(case when work_neighborhood_type = 'satellite_city' then p.cosine_similarity_afternoon else null end) as avg_cosine_similarity_work_afternoon_satellite_city,

# avg(case when work_neighborhood_type = poi_neighborhood_type then p.cosine_similarity_morning else null end) as avg_cosine_similarity_work_morning_same_neighborhood_type,
# avg(case when work_neighborhood_type = poi_neighborhood_type then p.cosine_similarity_late_morning else null end) as avg_cosine_similarity_work_late_morning_same_neighborhood_type,
# avg(case when work_neighborhood_type = poi_neighborhood_type then p.cosine_similarity_afternoon else null end) as avg_cosine_similarity_work_afternoon_same_neighborhood_type

from `phd-habidatum.phd_models_results.place2vec_poi_work_neighborhood_similarity_scheme3` p
group by poi_id
)

select g.* except(avg_cosine_similarity_work_morning, avg_cosine_similarity_work_late_morning, avg_cosine_similarity_work_afternoon,rank_cosine_similarity_work_morning_in_primarytype, rank_cosine_similarity_work_late_morning_in_primarytype, rank_cosine_similarity_work_afternoon_in_primarytype,rank_cosine_similarity_work_morning_residential_remote_in_primarytype, rank_cosine_similarity_work_morning_residential_central_in_primarytype, rank_cosine_similarity_work_morning_cbd_in_primarytype, rank_cosine_similarity_work_morning_same_neighborhood_type_in_primarytype, rank_cosine_similarity_work_morning_satellite_city_in_primarytype, rank_cosine_similarity_work_late_morning_residential_remote_in_primarytype, rank_cosine_similarity_work_late_morning_residential_central_in_primarytype, rank_cosine_similarity_work_late_morning_cbd_in_primarytype, rank_cosine_similarity_work_late_morning_satellite_city_in_primarytype, rank_cosine_similarity_work_late_morning_same_neighborhood_type_in_primarytype ),
m.*except(poi_id),

case when m.avg_cosine_similarity_work_morning is not null then PERCENT_RANK() OVER (
  PARTITION BY g.primarytype
  ORDER BY m.avg_cosine_similarity_work_morning
) else null end as rank_cosine_similarity_work_morning,

case when m.avg_cosine_similarity_work_late_morning is not null then PERCENT_RANK() OVER (
  PARTITION BY g.primarytype
  ORDER BY m.avg_cosine_similarity_work_late_morning
) else null end as rank_cosine_similarity_work_late_morning,

case when m.avg_cosine_similarity_work_afternoon is not null then PERCENT_RANK() OVER (
  PARTITION BY g.primarytype
  ORDER BY m.avg_cosine_similarity_work_afternoon
) else null end as rank_cosine_similarity_work_afternoon

# PERCENT_RANK() OVER (
#   PARTITION BY g.primarytype
#   ORDER BY m.avg_cosine_similarity_work_morning_residential_remote
# ) as rank_cosine_similarity_work_morning_residential_remote_in_primarytype,
# PERCENT_RANK() OVER (
#   PARTITION BY g.primarytype
#   ORDER BY m.avg_cosine_similarity_work_morning_residential_central
# ) as rank_cosine_similarity_work_morning_residential_central_in_primarytype,
# PERCENT_RANK() OVER (
#   PARTITION BY g.primarytype
#   ORDER BY m.avg_cosine_similarity_work_morning_cbd
# ) as rank_cosine_similarity_work_morning_cbd_in_primarytype,
# PERCENT_RANK() OVER (
#   PARTITION BY g.primarytype
#   ORDER BY m.avg_cosine_similarity_work_morning_satellite_city
# ) as rank_cosine_similarity_work_morning_satellite_city_in_primarytype,
# PERCENT_RANK() OVER (
#   PARTITION BY g.primarytype
#   ORDER BY m.avg_cosine_similarity_work_morning_same_neighborhood_type
# ) as rank_cosine_similarity_work_morning_same_neighborhood_type_in_primarytype,

# PERCENT_RANK() OVER (
#   PARTITION BY g.primarytype
#   ORDER BY m.avg_cosine_similarity_work_late_morning_residential_remote
# ) as rank_cosine_similarity_work_late_morning_residential_remote_in_primarytype,
# PERCENT_RANK() OVER (
#   PARTITION BY g.primarytype
#   ORDER BY m.avg_cosine_similarity_work_late_morning_residential_central
# ) as rank_cosine_similarity_work_late_morning_residential_central_in_primarytype,
# PERCENT_RANK() OVER (
#   PARTITION BY g.primarytype
#   ORDER BY m.avg_cosine_similarity_work_late_morning_cbd
# ) as rank_cosine_similarity_work_late_morning_cbd_in_primarytype,
# PERCENT_RANK() OVER (
#   PARTITION BY g.primarytype
#   ORDER BY m.avg_cosine_similarity_work_late_morning_satellite_city
# ) as rank_cosine_similarity_work_late_morning_satellite_city_in_primarytype,
# PERCENT_RANK() OVER (
#   PARTITION BY g.primarytype
#   ORDER BY m.avg_cosine_similarity_work_late_morning_same_neighborhood_type
# ) as rank_cosine_similarity_work_late_morning_same_neighborhood_type_in_primarytype

from `phd-habidatum.phd_pois.google_pois_features` g
left join m
on poi_id = id
""")





### LETS SEE WHAT VISIT VALUES HAVE POIS with KEYWORDS

In [ ]:
df_valid = utils.read_query_to_dataframe("select * from `phd-habidatum.phd_pois.google_pois_features`")

In [ ]:
diff = df_valid.select_dtypes(exclude='object').groupby(df_valid['flag_keyword_review_rw']).mean().T

In [ ]:
diff['diff']=(diff[1]-diff[0])/diff[0]
diff.sort_values(by='diff', ascending=False)

In [ ]:
diff.to_csv('diff_gtruth_others.csv')

In [ ]:
import seaborn as sns
sns.scatterplot(x='avg_rw_visits_share_after2022', y='avg_dist_home', data=df_valid[df_valid.flag_keyword_review_rw==0])

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

label = "flag_keyword_review_rw"  # or "flag_groundtruth"

metrics = [
    "avg_rw_visits_share_after2022",
    "avg_dist_home",
    "avg_dist_home_work_ratio",
    "visit_frequency",
    "share_dwells_home_lt_1km",
    "share_weekday",
    "share_friday",
    "avg_duration_hours_weekday",
    "n_visits_weekday",
    "n_visits_weekday_rw",
    "avg_duration_hours_weekday_rw",
    "visit_frequency_rw",
    "avg_dist_home_rw",
    "share_dwells_home_lt_1km_rw",
    "remote_workers_visits_share_weekday",
    "avg_rw_visits_share_percentile_0_1",
    "flag_rw_visits_share_after_2022",
]

plot_df = df_valid[[label] + metrics][df_valid['flag_location_suits_rw']==1].dropna(subset=[label]).copy()
plot_df[label] = plot_df[label].map({
    0: "Other POIs",
    1: "RW-labeled POIs",
})

# Clip extreme values so the violins are readable in presentation slides.
for col in metrics:
    plot_df[col] = pd.to_numeric(plot_df[col], errors="coerce").astype(float)
    lo, hi = plot_df[col].quantile([0.01, 0.99])
    plot_df[col] = plot_df[col].clip(lower=lo, upper=hi)

long_df = plot_df.melt(
    id_vars=label,
    value_vars=metrics,
    var_name="metric",
    value_name="value",
).dropna(subset=["value"])

g = sns.FacetGrid(
    long_df, 
    col="metric", 
    col_wrap=4, 
    height=3, 
    aspect=1, 
    sharey=False
)
g.map_dataframe(
    sns.violinplot, 
    x=label, 
    y="value", 
    linewidth=1,
    inner="box",
    cut=0
)

g.set_titles("{col_name}")
g.set_axis_labels("", "Metric value")

for ax in g.axes.flat:
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["Other POIs", "RW-labeled POIs"], rotation=20, ha="right")

plt.tight_layout()